**Overview**  
This notebook describes the process of finding the best hyperparameters for Graph neural Network, acrhitecture of which was found on the first step

First, we import all the necessary modules and define device for calculations - it is GPU card (*cuda:0*)

In [1]:
import os
import pandas as pd
import numpy as np
import pickle
from functools import partial

from rdkit import Chem
from chemlib import Element

from sklearn.model_selection import KFold, train_test_split, StratifiedGroupKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
from torchmetrics.classification import Accuracy, Precision, Recall, F1Score

from torch.utils.data import DataLoader, TensorDataset
import torch
import torch.nn as nn
from torch_geometric.nn import EdgeConv, NNConv, EdgePooling, global_add_pool
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader

device = 'cuda:0'
from warnings import filterwarnings
from hyperopt import hp, tpe, Trials, fmin, STATUS_OK

C:\Users\Timur\anaconda3\envs\ChemInfo-Pytorch\Lib\site-packages\torch_geometric\typing.py:86: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: [WinError 127] Не найдена указанная процедура
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
C:\Users\Timur\anaconda3\envs\ChemInfo-Pytorch\Lib\site-packages\torch_geometric\typing.py:124: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: [WinError 127] Не найдена указанная процедура
  warnings.warn(f"An issue occurred while importing 'torch-sparse'. "


Get an absolute path, which will be helpful for training function

In [2]:
abs_path = os.path.abspath('')

Functions below define how to node features, edge index and edge features, as described in the notebok *1.Architecture_selection.ipynb*. Class *Custom_dataset* unites them all to create dataset for Pytorch-geometric

In [3]:
def load_data(path = abs_path):
    Data_df = pd.read_excel('Data.xlsx', sheet_name='Sheet1')
    with open('groups.pickle', 'rb') as inp:
        groups = pickle.load(inp)
    return Data_df, groups      
    
        

In [4]:
def make_x(df_line):
    atom_properties = []
    mol = Chem.MolFromSmiles(df_line['SMILES'])
    for atom in mol.GetAtoms():
        _el = Element(atom.GetSymbol())
        atom_properties.append([atom.GetAtomicNum(), atom.GetMass(), int(atom.GetIsAromatic()), atom.GetExplicitValence(), 
                                atom.GetImplicitValence(), atom.GetTotalValence(), 
                                atom.GetNumExplicitHs(), atom.GetNumImplicitHs(), atom.GetTotalNumHs(),
                                atom.GetDegree(), atom.GetTotalDegree(), atom.GetFormalCharge(),
                               int(atom.IsInRing()), _el.Electronegativity, _el.FirstIonization, _el.AtomicRadius, _el.SpecificHeat, df_line['CTAB concentration (mM)'], df_line['Additive concentration'],
                                df_line['CTAB/additive'], df_line['Temperature']
                               ])
        x = torch.tensor(atom_properties, dtype = torch.float32)
    return x
        
    

In [5]:
def make_edge_indices(df_line):
    mol = Chem.MolFromSmiles(df_line['SMILES'])
    start_atoms = []
    end_atoms = []
    for bond in mol.GetBonds():
        start_atoms.append(bond.GetBeginAtomIdx())
        end_atoms.append(bond.GetEndAtomIdx())
    return torch.tensor([start_atoms, end_atoms], dtype = torch.int64)

In [6]:
def make_edge_features(df_line):
    test_molecule = Chem.MolFromSmiles(df_line['SMILES'])
    #instantiate array with zeros that will be filled with ones where necessary
    edge_features = np.zeros([test_molecule.GetNumBonds(), 4])
    #iterating through bonds in molecule. 
    for i, bond in enumerate(test_molecule.GetBonds()):
        if bond.GetBondTypeAsDouble() == 1.0:
            edge_features[i,0] = 1
        elif bond.GetBondTypeAsDouble() == 2.0:
            edge_features[i,1] = 1
        elif bond.GetBondTypeAsDouble() == 3.0:
            edge_features[i, 2] = 1
        #there are no other types except aromatic - 4th position
        else:
            edge_features[i, 3] = 1
    return torch.tensor(edge_features, dtype = torch.float32)

In [7]:
class Custom_dataset(Dataset):
    def __init__(self, X, y):
        super().__init__()
        self.X_df = X
        self.y_series = y
    def get(self, idx):
        instance = self.X_df.iloc[idx, :]
        try:
            x = make_x(instance)
        except:
            print('Invalid mol - cannot make x')
        try:
            edge_index = make_edge_indices(instance)
        except:
            print('Invalid mol - cannot make edge indices')
        try:
            edge_attr = make_edge_features(instance)
        except:
            print('Invalid mol - cannot make edge attrs')

        
        y = torch.tensor(self.y_series.iloc[idx], dtype = torch.long)
        data = Data(x = x, edge_index = edge_index, edge_attr = edge_attr, y = y)
        return data
    def len(self):
        return (len(self.X_df))

This is the best architecture, which was found earlier, on the first step. During the search of hyperparameters we will be chaning the number of neurons on each Graph Convolutional Layer (*n1* and *n2*)

In [8]:
class Graph_nn(nn.Module):
    def __init__(self, num_node_features = 21, n1 = 500, n2 = 500):
        super().__init__()
        self.nn_conv_1 = NNConv(in_channels = num_node_features, out_channels = n1, nn = nn.Sequential(nn.Linear(4, n1), nn.CELU(), nn.Linear(n1, 21*n1)))
        self.a_nn_conv1 = nn.CELU()
        
        self.edge_conv_3 = EdgeConv(nn = nn.Sequential(nn.Linear(2*n1, n2), nn.CELU(), nn.Linear(n2,n2)))
        self.a3 = nn.CELU()
        
        self.edgepool = EdgePooling(in_channels = n2,  dropout = 0.3) 
        self.linear_1 = nn.Linear(in_features = n2, out_features = 2)

        
    def forward(self, data):
        x, edge_index, edge_attr  = data.x, data.edge_index, data.edge_attr
        x = self.a_nn_conv1(self.nn_conv_1(x = x, edge_index = edge_index, edge_attr = edge_attr))
        x = self.a3(self.edge_conv_3(x = x, edge_index = edge_index))
       
        if data.batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        else:
            batch = data.batch
 
        x, new_edge_index, batch_assign, _ = self.edgepool(x = x, edge_index = edge_index, batch = batch)
        x = global_add_pool(x, batch = batch_assign)
        x = self.linear_1(x)
        return x

Here we check, that model is instantiated well

In [9]:
model = Graph_nn()

Thefunction *cross-valid-NN* is a key to all the work done. As there is no convenient cross-validation method for PyTorch, such as implemented in Sklearn, we create our own function to perform it. It takes several steps:
1. Loads data in form of Pandas DataFrame datasets and splits it into main (X, y, groups) and "untouchable" test (X_test, y_test, groups_test) datasets
2. Makes a StratifiedKfold;
3. Creates empty lists to store values for classification metrics for each fold;
4. Then it performs training and evaluation for each fold in KFold, using two functions, that were defined above;
5. Metric for each fold are stored in lists;
6. Function returns mean value of lists for each metric.

Parameters, that define hyperparameters of NN architecture (*n1*, *n2*) and training parameters (*learning rate*, *epochs*) are varied

In [9]:
def cross_valid_NN(path = abs_path, lr = 0.001, n1 = 2048, n2 = 2048, epochs = 10):
    f1_valid_history = []
    precision_valid_history = []
    accuracy_valid_history = []
    recall_valid_history = []
    Data_df, groups = load_data()
    Data_df.drop(columns = ['Trivial name', 'Zero-shear viscosity'], inplace = True)
    #df_train - for validation, df_test - for the special case
    df_train, df_test, groups_train, groups_test = train_test_split(Data_df, groups, random_state=0)
    kfold = StratifiedGroupKFold(n_splits = 5, shuffle = True, random_state = 0)
    torch.manual_seed(0)
    for split in kfold.split(X = df_train, y = df_train['Is gel'], groups = groups_train):
        #creating ds and dls for validation and training loops for every fold of cross-validation
        train_fold_ds = Custom_dataset(df_train.iloc[split[0]], df_train['Is gel'].iloc[split[0]])
        valid_fold_ds = Custom_dataset(df_train.iloc[split[1]], df_train['Is gel'].iloc[split[1]])
        
        train_fold_dl = DataLoader(train_fold_ds, batch_size=10, shuffle = True)
        valid_fold_dl = DataLoader(valid_fold_ds, batch_size=10, shuffle = True)
        #creating the model with specially designed parameters
        model = Graph_nn(n1 = n1, n2 = n2)
        model.to(device)
        #training loop for every fold
        model = train_model(train_fold_dl, model, lr, epochs)
        #model evaluation for every K-fold     
        model.eval()
        evaluation_results = evaluate_model(valid_fold_dl, model, threshold = 0.5)
        f1_valid_history.append(evaluation_results['f1'])
        precision_valid_history.append(evaluation_results['precision'])
        accuracy_valid_history.append(evaluation_results['accuracy'])
        recall_valid_history.append(evaluation_results['recall'])
    return np.mean(accuracy_valid_history), np.mean(f1_valid_history), np.mean(recall_valid_history), np.mean(precision_valid_history)




    

The first functions is used to train the model. It takes dataloader for training and perform the training loop for given number of epochs. The final result is the trained model

In [10]:
def train_model(train_dl, model, lr = 0.001, epochs = 10):
    model = model
    optimizer = torch.optim.Adam(params = model.parameters(), lr = lr, weight_decay = 1e-4)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(epochs):
            model.train()
            for batch in train_dl:
                optimizer.zero_grad()
                batch.to(device)
                output = model(batch)
                loss = loss_fn(output, batch.y)
                loss.backward()
                optimizer.step()
    return model

The nxt function is used to evaluate the model. It takes dataloader for the standard evaluation loop for all batches in dataloader. On this step we asess F1-score, precition, recall and accuracy

In [11]:
def evaluate_model(test_dl, model, threshold):
    valid_acc = Accuracy(task = 'binary')
    valid_precision = Precision(task = 'binary')
    valid_recall = Recall(task = 'binary')
    valid_f1 = F1Score(task = 'binary')
    ys = []
    predictions = []
    for batch in test_dl:
        batch.to(device)
        output = model(batch)
        predictions.append(output)
        ys.append(batch.y)
    predictions = torch.cat(predictions, dim = 0).cpu().detach()
    probs = torch.softmax(predictions, dim = 1)
    labels = (probs[:,1] > threshold).long()
    ys = torch.cat(ys).cpu()
    return {'f1':valid_f1(labels, ys).item(), 
            'precision':valid_precision(labels, ys).item(), 
            'recall':valid_recall(labels, ys).item(), 
            'accuracy':valid_acc(labels, ys).item()}

This function is used to get best parameters of the NN, that we defined above. It used hyperopt functions *fmin* to minimize the loss, which, in this case, is the **accuracy** of classification. During the search for optimal architecture and fine-tuning we found, that using **precision** as an loss, as we did for shallow ML methods, lead to the high rejection rate, which severely decreases accuracy to about 50%. 

In [12]:
def get_best_params(search_space = {'n1':hp.randint('n1', 1000),
                 'n2':hp.randint('n2', 1000),  
                 'lr':hp.uniform(label = 'lr', low = 10**(-5), high = 10**(-2)), 'epochs':hp.randint('epochs', 100)}, max_evals = 50):
    def objective(params):
        score = cross_valid_NN(n1 = params['n1'], n2 = params['n2'], lr = params['lr'], epochs = params['epochs'])
        return {'loss':-score[3], 'status':STATUS_OK, 'accuracy':score[3], 'f1':score[0], 'recall':score[2], 'precision':score[1]}
    trials = Trials()
    res = fmin(partial(objective), space = search_space, algo = tpe.suggest, trials = trials, max_evals = max_evals)
    return res, trials

In [13]:
best_params = get_best_params()

  0%|                                                                           | 0/50 [00:00<?, ?trial/s, best loss=?]

C:\Users\Timur\anaconda3\envs\ChemInfo-Pytorch\Lib\site-packages\torch_geometric\warnings.py:11: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(message)



  2%|▉                                            | 1/50 [09:06<7:26:27, 546.68s/trial, best loss: -0.7608500480651855]

C:\Users\Timur\anaconda3\envs\ChemInfo-Pytorch\Lib\site-packages\torch_geometric\warnings.py:11: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(message)



  4%|█▊                                          | 2/50 [27:46<11:47:00, 883.76s/trial, best loss: -0.7608500480651855]

...

 96%|██████████████████████████████████████████▏ | 48/50 [9:55:19<26:36, 798.36s/trial, best loss: -0.8031000375747681]

C:\Users\Timur\anaconda3\envs\ChemInfo-Pytorch\Lib\site-packages\torch_geometric\warnings.py:11: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(message)



 98%|██████████████████████████████████████████▏| 49/50 [10:15:33<15:23, 923.10s/trial, best loss: -0.8031000375747681]

C:\Users\Timur\anaconda3\envs\ChemInfo-Pytorch\Lib\site-packages\torch_geometric\warnings.py:11: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(message)



100%|███████████████████████████████████████████| 50/50 [10:20:51<00:00, 745.03s/trial, best loss: -0.8031000375747681]


Here we save the best hyperparameters, that we found

In [18]:
best_parameters = best_params[0]
print(best_parameters)
with open('best_params.pickle', 'wb') as out:
    pickle.dump(best_params, out)

{'epochs': 39, 'lr': 0.00012751361479088606, 'n1': 408, 'n2': 29}


The function *final_training_and_evaluation* was created, first, to assess the NN, which was trained using the optimal hyperparameter. For this task we use test dataset, which was not employed during cross-validation. The second task was to finally train the NN with the best hyperparameters and get the instance of it for final calculator

In [31]:
def final_training_and_evaluation(best_params, final = True):
    Data_df, groups = load_data()
    if final:
        ds = Custom_dataset(Data_df.drop(columns = 'Is gel'), Data_df['Is gel'])
        dl = DataLoader(ds, batch_size = 10)
    else:
        Df_train, Df_test = train_test_split(Data_df, random_state = 0)
        ds = Custom_dataset(Df_train, Df_train['Is gel'])
        ds_test = Custom_dataset(Df_test, Df_test['Is gel'])
        
        dl = DataLoader(ds, batch_size = 10)
        test_dl = DataLoader(ds_test, batch_size = 10)
        #dl = dl.to(device)
        #test_dl = test_dl.to(device)
    model = Graph_nn(n1 = best_parameters['n1'], n2 = best_parameters['n2'])
    model.to(device)
    model = train_model(dl, model, best_parameters['lr'], epochs = best_parameters['epochs'])
    if final:
        return model
    else:
        return(evaluate_model(test_dl, model, threshold = 0.5))

Let's assess the neural network using test dataset and NN, trained on the rest of the data

In [32]:
scores_standard = final_training_and_evaluation(best_params=best_parameters, final = False)

C:\Users\Timur\anaconda3\envs\ChemInfo-Pytorch\Lib\site-packages\torch_geometric\warnings.py:11: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(message)


In [33]:
scores_standard

{'f1': 0.8095238208770752,
 'precision': 0.7816091775894165,
 'recall': 0.8395061492919922,
 'accuracy': 0.7894737124443054}

We can see, that metrics, obtained for test dataset are realtively high, much higher than for Fully-connected NN. Therefore, we can save the best model in form of a *.pickle* file

In [34]:
model_standard = final_training_and_evaluation(best_parameters, final = True)

C:\Users\Timur\anaconda3\envs\ChemInfo-Pytorch\Lib\site-packages\torch_geometric\warnings.py:11: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(message)


In [35]:
with open('model_standard_acc.pickle', 'wb') as out:
    pickle.dump(model_standard, out)